### Step 1: Upload your Excel file

Please upload your Excel file to your Colab environment (e.g., by dragging and dropping it into the files pane on the left)

In [30]:
import pandas as pd

# --- IMPORTANT: Update the filename below to your uploaded Excel file's name ---
excel_filename = '/content/hall_of_fame_merged.xlsx' # e.g., 'baseball_hof.xlsx'

import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd # Ensure pandas is imported for pd.notna

# Ensure df_hof and PLAYER_NAME_COLUMN are defined (from previous cells)
if 'df_hof' in locals() and 'PLAYER_NAME_COLUMN' in locals():
    # Get all columns except the player name column for the second dropdown
    all_columns = df_hof.columns.tolist()
    # Exclude columns that are likely to be empty for all players or are control columns
    # 'Votes' and '% of Ballots' can be null if not inducted via BBWAA
    # 'ASG', 'WAR/pos', 'G', 'PA', etc. are typically for players only
    # Assuming all other columns are potential 'info_columns'
    initial_info_columns = [col for col in all_columns if col != PLAYER_NAME_COLUMN]
    initial_info_columns.sort() # Sort for consistent order


    # Get unique values for 'Inducted As' for the new 'Hall of Famer Type' dropdown
    inducted_as_options = df_hof['Inducted As'].unique().tolist()
    inducted_as_options.sort() # Sort for better readability
    inducted_as_options.insert(0, 'All') # Add an 'All' option to show all types

    if not initial_info_columns:
        print("No other information columns found besides the player name.")
    else:
        # Create the 'Hall of Famer Type' dropdown
        inducted_as_selector = widgets.Dropdown(
            options=inducted_as_options,
            value='All', # Default to 'All'
            description='Hall of Famer Type:', # Renamed description
            disabled=False,
        )

        # Initialize player_selector (options will be updated dynamically)
        player_selector = widgets.Dropdown(
            options=[], # Start empty, will be populated by update_player_options
            description='Select Player:',
            disabled=False,
        )

        # Create the information type dropdown
        info_selector = widgets.Dropdown(
            options=[], # Start empty, will be populated by update_info_options
            description='Select Info:',
            disabled=False,
        )

        # Output widget to display player info dynamically
        # Using the same output_widget from the previous cell if it exists, otherwise create a new one
        if 'output_widget' not in locals():
            output_widget = widgets.Output()

        # Main function to display player information based on current selections
        def update_display(change=None):
            with output_widget:
                clear_output(wait=True) # Clear previous output

                selected_player = player_selector.value
                selected_inducted_as = inducted_as_selector.value
                selected_info_column = info_selector.value

                # Only proceed if a player and an info column are selected
                if not selected_player or not selected_info_column:
                    print("Please make selections from all dropdowns.")
                    return

                print(f"\nDetails for: {selected_player}")

                # Start with the full DataFrame and apply filters for display
                filtered_df_for_display = df_hof.copy()

                # Apply 'Hall of Famer Type' filter to the data being displayed
                if selected_inducted_as != 'All':
                    filtered_df_for_display = filtered_df_for_display[filtered_df_for_display['Inducted As'] == selected_inducted_as]

                # Filter for the selected player from the *already filtered* display DataFrame
                player_data = filtered_df_for_display[filtered_df_for_display[PLAYER_NAME_COLUMN] == selected_player]

                if not player_data.empty:
                    # Display only the selected information column
                    # Ensure the selected_info_column actually exists in player_data before displaying
                    if selected_info_column in player_data.columns:
                        display(player_data[[PLAYER_NAME_COLUMN, selected_info_column]])
                    else:
                        print(f"Selected info column '{selected_info_column}' not available for this player.")
                else:
                    print("Player data not found for the selected criteria. Please make sure a player is selected.")

        # Function to update the options of the info_selector based on selected player
        def update_info_options(selected_player_name):
            # Handle cases where selected_player_name might be empty (e.g., if no players for a category)
            if not selected_player_name:
                info_selector.options = []
                info_selector.value = ''
                update_display()
                return

            # Get the row for the selected player
            # Use .head(1) in case of multiple matches, though unique names should prevent this
            player_row_df = df_hof[df_hof[PLAYER_NAME_COLUMN] == selected_player_name]
            if player_row_df.empty:
                info_selector.options = []
                info_selector.value = ''
                update_display()
                return
            player_row = player_row_df.iloc[0]

            # Filter initial_info_columns to only include those with non-null values for the player
            available_info_columns = [col for col in initial_info_columns if pd.notna(player_row[col])]
            available_info_columns.sort()

            # Temporarily unobserve info_selector to prevent immediate trigger
            info_selector.unobserve(update_display, names='value')

            info_selector.options = available_info_columns
            if available_info_columns:
                # If current value is not in new options, reset to first available
                if info_selector.value not in available_info_columns:
                    info_selector.value = available_info_columns[0]
            else:
                info_selector.value = '' # No info columns available

            # Re-observe info_selector
            info_selector.observe(update_display, names='value')

            # Trigger display update after info options are set
            update_display()

        # Define a named function for player_selector to update info_options
        def _player_changed_update_info(change):
            update_info_options(change['new'])

        # Function to update the options of the player_selector based on 'Hall of Famer Type' selection
        def update_player_options_and_trigger_display(change):
            selected_inducted_as_type = change['new']

            # Filter df_hof to get players for the dropdown options
            if selected_inducted_as_type == 'All':
                players_for_dropdown = df_hof
            else:
                players_for_dropdown = df_hof[df_hof['Inducted As'] == selected_inducted_as_type]

            new_player_names = players_for_dropdown[PLAYER_NAME_COLUMN].unique().tolist()
            new_player_names.sort()

            # Temporarily unobserve player_selector to avoid immediate duplicate triggers
            player_selector.unobserve(update_display, names='value')
            # Use the named function for unobserving
            player_selector.unobserve(_player_changed_update_info, names='value')

            # Update player_selector options
            player_selector.options = new_player_names
            if new_player_names:
                # If the currently selected player is not in the new options, reset to the first option
                if player_selector.value not in new_player_names:
                    player_selector.value = new_player_names[0]
                # After updating player options and value, update info options for the new player
                update_info_options(player_selector.value)
            else:
                player_selector.value = '' # Set to empty if no players are found for the category
                info_selector.options = [] # Clear info options if no player
                info_selector.value = ''
                update_display() # Trigger display update even if no players/info

            # Re-observe player_selector for both update_display and _player_changed_update_info
            player_selector.observe(update_display, names='value')
            player_selector.observe(_player_changed_update_info, names='value')


        # Attach observers
        # 'Hall of Famer Type' dropdown changes update the player dropdown options
        inducted_as_selector.observe(update_player_options_and_trigger_display, names='value')
        # Player dropdown changes update the displayed information AND the info_selector options
        player_selector.observe(update_display, names='value')
        player_selector.observe(_player_changed_update_info, names='value') # Use the named function here
        # Info dropdown changes update the displayed information
        info_selector.observe(update_display, names='value')

        # Display all dropdowns and the output area in the desired order
        display(inducted_as_selector, player_selector, info_selector, output_widget)

        # Initial setup: update player selector options based on the default 'Hall of Famer Type' value
        # This will also trigger update_info_options and update_display implicitly.
        update_player_options_and_trigger_display({'new': inducted_as_selector.value})

else:
    print("Please run the previous cells to load data and define 'df_hof' and 'PLAYER_NAME_COLUMN'.")

Dropdown(description='Hall of Famer Type:', options=('All', 'Manager', 'Pioneer/Executive', 'Player', 'Umpire'…

Dropdown(description='Select Player:', options=(), value=None)

Dropdown(description='Select Info:', options=(), value=None)

Output()